<a href="https://colab.research.google.com/github/sebastiancasas10/curso-ia-para-economia/blob/main/2_Taller_Apriori_Sebastian_Casas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Nombre estudiante:**

- Sebastán Casas Poloche


**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [ ]:
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/dataset_taller_2/'
os.chdir(path)

In [ ]:
df = pd.read_csv('alimentos_retail_top30.csv', header=None)
df

,0,1,2,3,4,5,6,7
0,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,17452.0,Colombia
2,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,17779.0,Colombia
3,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,14933.0,Colombia
4,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,14957.0,Colombia
...,...,...,...,...,...,...,...,...
6895,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,13520.0,México
6896,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,12105.0,México
6897,537999,36355,TOMATE,-1,2023-01-07 20:26:00,4.87,16918.0,México
6898,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,15425.0,México


In [ ]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6900, 8)


In [ ]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6900 entries, 0 to 6899
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       6900 non-null   object
 1   1       6900 non-null   object
 2   2       6900 non-null   object
 3   3       6900 non-null   object
 4   4       6900 non-null   object
 5   5       6900 non-null   object
 6   6       6880 non-null   object
 7   7       6900 non-null   object
dtypes: object(8)
memory usage: 431.4+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [ ]:

df.isnull().sum()

,0
0,0
1,0
2,0
3,0
4,0
5,0
6,20
7,0


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [ ]:

df.describe()

,0,1,2,3,4,5,6,7
count,6900,6900,6900,6900,6900,6900,6880,6900
unique,2006,21,26,10,1799,20,4125,3
top,537005,74520,CHILE JALAPEÑO,4,2023-01-06 06:15:00,3.39,17950.0,México
freq,5,420,419,1432,14,541,8,3551


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

Al observar los resultados anteriores, se identifican los siguientes problemas:
- Se evidencian valores nulos (20 registros faltantes), lo que puede afectar el análisis, ya que no se puede identificar correctamente al cliente en esas transacciones.

- Se observan valores negativos, lo cual no es lógico en el contexto de compras normales. Esto puede indicar devoluciones, errores en el registro o datos inconsistentes.

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [ ]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
df_limpio = df.dropna(subset=['CustomerID'])
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,17452.0,Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,17779.0,Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,14933.0,Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,14957.0,Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,15202.0,Colombia
...,...,...,...,...,...,...,...,...
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,13520.0,México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,12105.0,México
6896,537999,36355,TOMATE,-1,2023-01-07 20:26:00,4.87,16918.0,México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,15425.0,México


In [ ]:
df.columns = df.iloc[0]
df = df[1:]
df = df.reset_index(drop=True)
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,17452.0,Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,17779.0,Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,14933.0,Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,14957.0,Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,15202.0,Colombia
...,...,...,...,...,...,...,...,...
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,13520.0,México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,12105.0,México
6896,537999,36355,TOMATE,-1,2023-01-07 20:26:00,4.87,16918.0,México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,15425.0,México


In [ ]:
# El tipo de dato de CustomerID debe ser entero
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(float).astype(int)


In [ ]:
df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6879 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   InvoiceNo    6879 non-null   object
 1   StockCode    6879 non-null   object
 2   Description  6879 non-null   object
 3   Quantity     6879 non-null   object
 4   InvoiceDate  6879 non-null   object
 5   UnitPrice    6879 non-null   object
 6   CustomerID   6879 non-null   int64 
 7   Country      6879 non-null   object
dtypes: int64(1), object(7)
memory usage: 483.7+ KB


2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [ ]:
# TAREA: # Verifica cuántas descripciones únicas hay.
df_limpio['Description'].nunique()


25

In [ ]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [ ]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
df_limpio['Description'].nunique()

20

2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [ ]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
df_limpio = df_limpio[~df_limpio['InvoiceNo'].str.startswith('C')]

In [ ]:
# TAREA: Elimina las filas con cantidades negativas.
df_limpio = df_limpio[df_limpio['Quantity'].astype(float) > 0]


In [ ]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [ ]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
df_limpio['Quantity'] = pd.to_numeric(df_limpio['Quantity'], errors='coerce')
cesta_mx = preparar_cesta(df_limpio, 'México')

In [ ]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
from mlxtend.frequent_patterns import apriori
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)
frequent_itemsets_mx.sort_values(by='support', ascending=False).head(10)

,support,itemsets
2,0.418,(CHILE JALAPEÑO)
7,0.413,(TOMATE)
3,0.405,(CILANTRO)
1,0.405,(CEBOLLA)
8,0.378,(TORTILLAS DE MAÍZ)
4,0.356,(FRIJOL NEGRO)
0,0.352,(AGUACATE)
5,0.352,(LIMÓN)
9,0.331,(TOTOPOS)
31,0.330,"(CHILE JALAPEÑO, TOMATE)"


In [ ]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
from mlxtend.frequent_patterns import association_rules
rules_mx = association_rules(frequent_itemsets_mx, metric="lift", min_threshold=2)
rules_mx.head()


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(LIMÓN),(AGUACATE),0.352,0.352,0.266,0.755682,2.146823,1.0,0.142096,2.652279,0.824376,0.607306,0.622966,0.755682
1,(AGUACATE),(LIMÓN),0.352,0.352,0.266,0.755682,2.146823,1.0,0.142096,2.652279,0.824376,0.607306,0.622966,0.755682
2,(TOTOPOS),(AGUACATE),0.331,0.352,0.262,0.791541,2.248695,1.0,0.145488,3.108522,0.830041,0.622328,0.678304,0.767929
3,(AGUACATE),(TOTOPOS),0.352,0.331,0.262,0.744318,2.248695,1.0,0.145488,2.616533,0.856941,0.622328,0.617815,0.767929
4,(TORTILLAS DE MAÍZ),(FRIJOL NEGRO),0.378,0.356,0.293,0.775132,2.177338,1.0,0.158432,2.863906,0.869330,0.664399,0.650827,0.799083


In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_mx.sort_values(by=['lift', 'confidence'], ascending=False)[[
    'antecedents',
    'consequents',
    'antecedent support',
    'consequent support',
    'confidence',
    'lift'
]].head(10)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
91,"(CEBOLLA, CHILE JALAPEÑO)","(CILANTRO, TOMATE)",0.322,0.317,0.919255,2.899857
90,"(CILANTRO, TOMATE)","(CEBOLLA, CHILE JALAPEÑO)",0.317,0.322,0.933754,2.899857
88,"(CILANTRO, CEBOLLA)","(CHILE JALAPEÑO, TOMATE)",0.311,0.330,0.951768,2.884147
93,"(CHILE JALAPEÑO, TOMATE)","(CILANTRO, CEBOLLA)",0.330,0.311,0.896970,2.884147
11,"(LIMÓN, AGUACATE)",(TOTOPOS),0.266,0.331,0.932331,2.816709
14,(TOTOPOS),"(LIMÓN, AGUACATE)",0.331,0.266,0.749245,2.816709
89,"(CILANTRO, CHILE JALAPEÑO)","(CEBOLLA, TOMATE)",0.322,0.327,0.919255,2.811176
92,"(CEBOLLA, TOMATE)","(CILANTRO, CHILE JALAPEÑO)",0.327,0.322,0.905199,2.811176
73,"(LIMÓN, TOMATE, AGUACATE)",(TOTOPOS),0.023,0.331,0.913043,2.758440
76,(TOTOPOS),"(LIMÓN, TOMATE, AGUACATE)",0.331,0.023,0.063444,2.758440


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

Las reglas con mayor lift muestran asociaciones muy fuertes entre productos típicos de la cocina mexicana. Por ejemplo, combinaciones como cebolla, chile jalapeño, cilantro y tomate aparecen frecuentemente juntas, lo que indica que los clientes compran estos productos en conjunto. Estas asociaciones reflejan ingredientes utilizados en preparaciones comunes como salsas o guacamole. En general, se trata de productos complementarios, es decir, que se consumen juntos como parte de una misma receta o plato.

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

En estas reglas se observa que el support indica que los productos aparecen con frecuencia considerable dentro de las transacciones (por ejemplo, valores cercanos a 0.3 muestran que están presentes en aproximadamente el 30% de las compras).

 La confidence, cercana a valores altos (mayores a 0.9), indica que cuando se compran los productos del antecedente, es muy probable que también se compren los del consecuente.

 Finalmente, el lift mayor a 2 demuestra que la relación entre estos productos es fuerte y no ocurre por azar, sino que existe una asociación real en el comportamiento de compra de los clientes.

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

Basado en estas reglas, se pueden diseñar estrategias de venta como promociones de productos combinados, como lo son combos de ingredientes para preparar salsas o guacamole. También se podrían ubicar estos productos juntos en el punto de venta para incentivar compras cruzadas.

Otra estrategia sería ofrecer descuentos al comprar varios de estos productos en conjunto, aprovechando que los clientes tienden a adquirirlos simultáneamente.

3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.

cesta_co = preparar_cesta(df_limpio, 'Colombia')

In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
from mlxtend.frequent_patterns import apriori

frequent_itemsets_co = apriori(cesta_co, min_support=0.02, use_colnames=True)
frequent_itemsets_co.sort_values(by='support', ascending=False).head(10)


,support,itemsets
3,0.414,(CAFÉ)
4,0.407,(FRIJOL CARGAMANTO)
0,0.402,(ACEITE DE GIRASOL)
2,0.398,(AZÚCAR)
7,0.392,(LECHE)
1,0.379,(ARROZ)
9,0.346,(QUESO MUZZARELLA)
5,0.338,(HARINA DE MAÍZ)
37,0.323,"(LECHE, CAFÉ)"
31,0.317,"(LECHE, AZÚCAR)"


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
from mlxtend.frequent_patterns import association_rules

rules_co = association_rules(frequent_itemsets_co, metric="lift", min_threshold=2)

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_co.sort_values(by=['lift', 'confidence'], ascending=False)[[
    'antecedents',
    'consequents',
    'antecedent support',
    'consequent support',
    'confidence',
    'lift'
]].head(10)


,antecedents,consequents,antecedent support,consequent support,confidence,lift
42,"(FRIJOL CARGAMANTO, CAFÉ, AZÚCAR)",(LECHE),0.046,0.392,0.956522,2.440106
43,(LECHE),"(FRIJOL CARGAMANTO, CAFÉ, AZÚCAR)",0.392,0.046,0.112245,2.440106
12,"(CAFÉ, AZÚCAR)",(LECHE),0.316,0.392,0.946203,2.413782
13,(LECHE),"(CAFÉ, AZÚCAR)",0.392,0.316,0.762755,2.413782
4,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.301,0.379,0.913621,2.410610
9,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.379,0.301,0.725594,2.410610
59,"(PAN TAJADO, CAFÉ, AZÚCAR)",(LECHE),0.031,0.392,0.935484,2.386438
66,(LECHE),"(PAN TAJADO, CAFÉ, AZÚCAR)",0.392,0.031,0.073980,2.386438
50,"(HUEVOS, CAFÉ, AZÚCAR)",(LECHE),0.040,0.392,0.925000,2.359694
55,(LECHE),"(HUEVOS, CAFÉ, AZÚCAR)",0.392,0.040,0.094388,2.359694


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

Las reglas con mayor lift en Colombia muestran un patrón claro de consumo alrededor de productos básicos y de desayuno, especialmente combinaciones como café, azúcar, leche y pan. Por ejemplo, se observa que productos como café y azúcar están fuertemente asociados con la compra de leche, lo que indica hábitos de consumo típicos como el desayuno o bebidas calientes. Asimismo, combinaciones como frijol cargamanto, aceite y arroz reflejan patrones de alimentación tradicionales del mercado colombiano.

A diferencia de México, donde predominaban ingredientes frescos para preparaciones como salsas (cilantro, tomate, cebolla), en Colombia se evidencia un consumo más orientado a productos básicos y acompañamientos cotidianos.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

En estas reglas, el support indica que los productos tienen una presencia significativa en las transacciones (por ejemplo, valores cercanos a 0.3 muestran que aparecen en aproximadamente el 30% de las compras).

La confidence, que en muchos casos supera el 90%, indica una alta probabilidad de que al comprar los productos del antecedente, también se adquieran los del consecuente.

Por su parte, el lift, superior a 2, demuestra que estas asociaciones son fuertes y no ocurren por azar, lo que confirma patrones de compra consistentes en el comportamiento del consumidor colombiano.

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

Para el mercado colombiano, se podrían diseñar estrategias enfocadas en combos de productos básicos, especialmente relacionados con el desayuno, como paquetes de café, azúcar y leche. También sería recomendable implementar promociones cruzadas entre productos como arroz, frijol y aceite, que suelen consumirse juntos en preparaciones tradicionales.

A diferencia de México, donde las promociones pueden centrarse en ingredientes frescos, en Colombia es más efectivo enfocar las estrategias en productos de consumo diario y de alta rotación.